# Jersey colour classification in football video


In [37]:
import torch
import pandas as pd
import numpy as np


from dataset import load_manifest, get_loader
from models import load_model
from extract_embeddings import get_embeddings
from classification_clustering import run_classification, run_clustering
from visualization import interactive_embedding_view, plot_confusion_matrix
from metrics import crop_accuracy, crop_macro_f1, clustering_accuracy, silhouette_scores

from sklearn.model_selection import train_test_split

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
manifest_path = "dataset_v1/manifest_with_splits.csv"

df = load_manifest(manifest_path)

print("dataset size:", len(df))
df.head()

dataset size: 64291


,crop_path,label,game,src_image,x1,y1,x2,y2,frame_idx,player_id,role_name,left2right,split
0,dataset_v1/crops/game_10_H1__frame_001877__i0_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,1399,522,1475,667,1877,101,Left Central Back,1,train
1,dataset_v1/crops/game_10_H1__frame_001877__i1_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,1524,290,1591,408,1877,103,Right Central Back,1,train
2,dataset_v1/crops/game_10_H1__frame_001877__i2_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,1091,874,1171,1067,1877,104,Left Back,1,train
3,dataset_v1/crops/game_10_H1__frame_001877__i3_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,77,549,126,704,1877,105,Left Midfielder,1,train
4,dataset_v1/crops/game_10_H1__frame_001877__i4_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,185,169,223,269,1877,106,Right Winger,1,train


In [3]:
df["game"].unique()[:20]

<StringArray>
['game_10_H1', 'game_10_H2', 'game_11_H1', 'game_11_H2', 'game_12_H1',
 'game_12_H2', 'game_13_H1', 'game_13_H2', 'game_14_H1', 'game_14_H2',
 'game_15_H1', 'game_15_H2', 'game_16_H1', 'game_16_H2', 'game_17_H1',
 'game_17_H2', 'game_19_H1', 'game_19_H2',  'game_1_H1',  'game_1_H2']
Length: 20, dtype: str

## game_12


In [25]:
base_game = "game_12"
df_h1 = df[df["game"] == f"{base_game}_H1"].copy()
df_h2 = df[df["game"] == f"{base_game}_H2"].copy()

print("H1:", len(df_h1), "H2:", len(df_h2), "Total:", len(df_h1) + len(df_h2))

H1: 716 H2: 695 Total: 1411


In [26]:
def flip_lr_label(label: str) -> str:
    if label == "team_left":
        return "team_right"
    if label == "team_right":
        return "team_left"
    return label

df_h2["label"] = df_h2["label"].map(flip_lr_label)

df_match = pd.concat([df_h1, df_h2], ignore_index=True)

print(df_match["label"].value_counts())

label
team_right    680
team_left     636
goalkeeper     95
Name: count, dtype: int64


In [27]:
loader = get_loader(df_match, batch_size=128, model_name="osnet")
model = load_model("osnet", device)

X_match_osnet, y_match_osnet = get_embeddings(
    f"osnet_{base_game}",
    model,
    loader
)

d:\Anaconda\envs\footpass\Lib\site-packages\torchreid\reid\models\osnet.py:482: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(cached_file)


Successfully loaded imagenet pretrained weights from "C:\Users\Антон/.cache\torch\checkpoints\osnet_x1_0_imagenet.pth"
computing embeddings


  0%|          | 0/12 [00:00<?, ?it/s]c:\Users\Антон\Desktop\Skoltech\MSc\ML\Project\models.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=self.device == "cuda"):
100%|██████████| 12/12 [00:09<00:00,  1.23it/s]


In [28]:
loader_dino = get_loader(df_match, batch_size=128, model_name="dino")
model_dino = load_model("dino", device)

X_match_dino, y_match_dino = get_embeddings(
    f"dino_{base_game}",
    model_dino,
    loader_dino
)

d:\Anaconda\envs\footpass\Lib\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name vit_base_patch16_224_dino to current vit_base_patch16_224.dino.
  model = create_fn(


computing embeddings


100%|██████████| 12/12 [00:13<00:00,  1.13s/it]


### Metrics

In [29]:
# Osnet

X_train, X_test, y_train, y_test = train_test_split(X_match_osnet, y_match_osnet, test_size=0.2, random_state=42)
cls_results, y_pred = run_classification(
    X_train,
    y_train,
    X_test,
    y_test
)

print("classification results")
print(cls_results)

classification results
{'accuracy': 0.9611307420494699, 'macro_f1': 0.935591046454424}


In [30]:
clust_results, clusters = run_clustering(
    X_match_osnet,
    y_match_osnet
)

print("clustering results")
print(clust_results)

clustering results
{'clustering_accuracy': np.float64(0.7512402551381998), 'silhouette_euclidean': 0.08316767181276483, 'silhouette_cosine': 0.15183141641779108}


In [31]:
# Dino

X_train, X_test, y_train, y_test = train_test_split(X_match_dino, y_match_dino, test_size=0.2, random_state=42)
cls_results, y_pred = run_classification(
    X_train,
    y_train,
    X_test,
    y_test
)

print("classification results")
print(cls_results)

classification results
{'accuracy': 0.950530035335689, 'macro_f1': 0.9381820918523992}


In [32]:
clust_results, clusters = run_clustering(
    X_match_dino,
    y_match_dino
)

print("clustering results")
print(clust_results)

clustering results
{'clustering_accuracy': np.float64(0.7370659107016301), 'silhouette_euclidean': 0.0924452394247055, 'silhouette_cosine': 0.16468502581119537}


### Visualization

In [34]:
df_viz = df_match[["crop_path", "game", "frame_idx", "player_id"]].reset_index(drop=True)

In [36]:
# OSNet

interactive_embedding_view(
    X=X_match_osnet,
    y=y_match_osnet,
    df=df_viz,
    method="umap",
    sample_n=None,
    preload_images=True,
)


Снижение размерности (UMAP) для 1411 точек...


d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Готово.
Кодирование изображений...
Готово.


    'data': [{'customdata': array([['team_left', 'game_12_H1', '692', '101',
   …

FigureWidget({
    'data': [{'customdata': array([['team_left', 'game_12_H1', '692', '101',
                                    '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0dHx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh7/wAARCABHACgDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDHopSKB15rwbHCJgEVBcgMjIXYICN7K2NoJ9e1dJ4

In [12]:
# Dino

interactive_embedding_view(
    X=X_match_dino,
    y=y_match_dino,
    df=df_viz,
    method="umap",
    sample_n=None,
    preload_images=True,
)

Снижение размерности (UMAP) для 716 точек...


d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Готово.
Кодирование изображений...
Готово.


    'data': [{'customdata': array([['team_left', 'game_12_H1', '692', '101',
   …

FigureWidget({
    'data': [{'customdata': array([['team_left', 'game_12_H1', '692', '101',
                                    '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0dHx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh7/wAARCABHACgDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDHopSKB15rwbHCJgEVBcgMjIXYICN7K2NoJ9e1dJ4